# Legally AI — build the FAISS index on Colab (Phase 0, scaled)

Downloads a slice of the **Indian Supreme Court** corpus (AWS Open Data, CC-BY-4.0),
chunks + embeds each judgment with **InLegalBERT** on GPU, and produces:

- `corpus.faiss` — the vector index
- `corpus_meta.parquet` — parallel metadata (row order == faiss ids)

**Before running:** Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save. Then Runtime ▸ Run all.

Main knob: `LIMIT_PER_YEAR` in the config cell.


In [ ]:
# 1. GPU check + dependencies
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > T4 GPU'
!pip install -q boto3 pymupdf pypdf pandas pyarrow transformers torch faiss-cpu numpy


## 2. Write the pipeline scripts (verbatim from the repo)


In [ ]:
%%writefile /content/download_corpus.py
"""Phase 0 — download a narrow slice of the Indian SC judgment corpus (brief §5).

Source: "Indian Supreme Court Judgments" on the AWS Open Data Registry
(CC-BY-4.0), maintained by the Open Justice India project
(github.com/vanga/indian-supreme-court-judgments). Public bucket, anonymous
(unsigned) access.

Bucket layout (verified):
  metadata/parquet/year=YYYY/metadata.parquet     <- structured metadata
  data/pdf/year=YYYY/english/{path}_EN.pdf         <- per-judgment English PDF
  data/tar/year=YYYY/english/english.tar           <- whole-year text archive

Parquet columns: title, petitioner, respondent, description, judge,
  author_judge, citation, case_id, cnr, decision_date, disposal_nature,
  court, available_languages, raw_html, path, nc_display, scraped_at, year.

This script reads the parquet metadata and, for up to --limit judgments per
year, downloads the individual English PDF and extracts its text, writing one
normalized JSON per judgment to data/raw/ for chunk_embed.py.

Per-PDF download keeps small slices cheap. For the FULL corpus, prefer the
year `english.tar` archives and run embedding on free Kaggle/Colab GPU (brief §3).

Usage:
  python download_corpus.py --start-year 2015 --end-year 2024 --limit 25
"""
from __future__ import annotations

import argparse
import io
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("download_corpus")

ROOT = Path(__file__).resolve().parent.parent
RAW_DIR = ROOT / "data" / "raw"

BUCKET = "indian-supreme-court-judgments"
ATTRIBUTION = "CC-BY-4.0 — Indian Supreme Court Judgments, Open Justice India (vanga), via AWS Open Data."


def _s3():
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config

    return boto3.client("s3", config=Config(signature_version=UNSIGNED))


def _read_parquet(s3, year: int):
    import pandas as pd

    key = f"metadata/parquet/year={year}/metadata.parquet"
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(io.BytesIO(obj["Body"].read()))


def _extract_pdf_text(data: bytes) -> str:
    """Extract text from PDF bytes; PyMuPDF first, pypdf fallback."""
    try:
        import fitz  # PyMuPDF

        with fitz.open(stream=data, filetype="pdf") as doc:
            return "\n".join(page.get_text() for page in doc).strip()
    except Exception:  # noqa: BLE001 - fall back to pypdf
        try:
            from pypdf import PdfReader

            reader = PdfReader(io.BytesIO(data))
            return "\n".join((p.extract_text() or "") for p in reader.pages).strip()
        except Exception as e:  # noqa: BLE001
            log.warning("    PDF text extraction failed: %s", e)
            return ""


def _normalize_outcome(disposal_nature: str) -> str:
    """Keep the raw disposal label but tag a coarse binary where obvious."""
    d = (disposal_nature or "").lower()
    if "allow" in d:
        return f"{disposal_nature} (Granted)"
    if "dismiss" in d:
        return f"{disposal_nature} (Dismissed)"
    return disposal_nature or "Unknown"


def main() -> None:
    ap = argparse.ArgumentParser(description="Download SC judgment corpus slice.")
    ap.add_argument("--start-year", type=int, default=2015)
    ap.add_argument("--end-year", type=int, default=2024)
    ap.add_argument("--limit", type=int, default=25, help="max judgments PER YEAR")
    ap.add_argument("--min-chars", type=int, default=500, help="skip docs with less text")
    ap.add_argument("--out", type=Path, default=RAW_DIR)
    args = ap.parse_args()

    args.out.mkdir(parents=True, exist_ok=True)
    s3 = _s3()
    manifest = {"bucket": BUCKET, "license": ATTRIBUTION, "years": {}, "files": []}
    total = 0

    for year in range(args.start_year, args.end_year + 1):
        try:
            df = _read_parquet(s3, year)
        except Exception as e:  # noqa: BLE001
            log.warning("year %d: no metadata (%s), skipping", year, e)
            continue

        log.info("year %d: %d judgments in metadata; taking up to %d", year, len(df), args.limit)
        taken = 0
        for _, row in df.iterrows():
            if taken >= args.limit:
                break
            path = str(row.get("path") or "").strip()
            if not path:
                continue
            pdf_key = f"data/pdf/year={year}/english/{path}_EN.pdf"
            try:
                obj = s3.get_object(Bucket=BUCKET, Key=pdf_key)
                text = _extract_pdf_text(obj["Body"].read())
            except Exception as e:  # noqa: BLE001
                log.warning("    %s: download/extract failed (%s)", pdf_key, e)
                continue
            if len(text) < args.min_chars:
                continue

            doc = {
                "case_name": str(row.get("title") or ""),
                "citation": str(row.get("citation") or ""),
                "neutral_citation": str(row.get("case_id") or ""),
                "court": str(row.get("court") or "Supreme Court of India"),
                "year": int(year),
                "decision_date": str(row.get("decision_date") or ""),
                "judges": str(row.get("judge") or ""),
                "outcome": _normalize_outcome(str(row.get("disposal_nature") or "")),
                "source_pdf": f"s3://{BUCKET}/{pdf_key}",
                "full_text": text,
            }
            out_path = args.out / f"{year}_{path}.json"
            out_path.write_text(json.dumps(doc, ensure_ascii=False, indent=2), encoding="utf-8")
            manifest["files"].append(out_path.name)
            taken += 1
            total += 1
            log.info("    [%d] %s", total, doc["case_name"][:70])

        manifest["years"][str(year)] = taken

    (args.out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    log.info("Done. Wrote %d judgments to %s", total, args.out)
    if total == 0:
        raise SystemExit("No judgments downloaded — check network/year range.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/chunk_embed.py
"""Phase 0 — role-aware chunk + embed + build FAISS index (brief §5).

Reads raw judgments from data/raw/, segments each into rhetorical roles
(Facts, Issues, Arguments, Ratio/Holding, ...), embeds chunks with InLegalBERT
(SAME model + mean-pool + L2-norm as backend/embeddings.py), and writes:
  * data/index/corpus.faiss        (IndexFlatIP over normalized vectors == cosine)
  * data/index/corpus_meta.parquet (parallel metadata, row order == faiss ids)

Run OFFLINE on free Kaggle/Colab GPU. The backend only reads the artifacts.

Why role-aware chunking: the Ratio/Holding segments carry the decided outcome
and reasoning; retrieval.py up-weights them. Don't chunk blindly by tokens.
"""
from __future__ import annotations

import os

# Permit PyTorch + FAISS dual OpenMP runtimes (Windows "OMP: Error #15").
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import argparse
import json
import re
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
RAW_DIR = ROOT / "data" / "raw"
INDEX_DIR = ROOT / "data" / "index"

EMBEDDING_MODEL = "law-ai/InLegalBERT"
EMBEDDING_DIM = 768

# Heuristic rhetorical-role markers for Indian judgments. Replace with a trained
# rhetorical-role classifier later for better segmentation.
ROLE_MARKERS = {
    "Facts": [r"\bfacts?\b", r"\bbrief facts\b", r"\bfactual matrix\b"],
    "Issues": [r"\bissues?\b", r"\bquestions? of law\b", r"\bpoints? for determination\b"],
    "Arguments": [r"\bsubmissions?\b", r"\bcontend(?:ed|s)?\b", r"\blearned counsel\b"],
    "Ratio": [r"\bwe are of the (?:view|opinion)\b", r"\bratio\b", r"\bheld\b", r"\bwe hold\b"],
    "Holding": [r"\baccordingly\b", r"\bappeal is (?:allowed|dismissed)\b", r"\bdisposed of\b"],
}


def _load_embedder():
    import torch
    from transformers import AutoModel, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
    model = AutoModel.from_pretrained(EMBEDDING_MODEL)
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    return tok, model, torch, device


def _embed(texts: list[str], tok, model, torch, device, batch_size: int = 32) -> np.ndarray:
    vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(
                texts[i : i + batch_size],
                padding=True, truncation=True, max_length=512, return_tensors="pt",
            ).to(device)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).type_as(hidden)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            vecs.append(pooled.cpu().numpy().astype("float32"))
    return np.vstack(vecs)


def _label_role(text: str) -> str:
    low = text.lower()
    for role, pats in ROLE_MARKERS.items():
        if any(re.search(p, low) for p in pats):
            return role
    return "Body"


def _split_long(text: str, max_chars: int) -> list[str]:
    """Hard-split an over-long block into <=max_chars pieces, preferring a
    sentence/word boundary. Needed because many SC PDFs extract with NO blank
    lines, so paragraph splitting alone leaves one giant chunk per judgment."""
    pieces: list[str] = []
    while len(text) > max_chars:
        window = text[:max_chars]
        cut = max(window.rfind(". "), window.rfind("\n"), window.rfind(" "))
        if cut <= 0:
            cut = max_chars  # no boundary found; cut hard
        pieces.append(text[: cut + 1].strip())
        text = text[cut + 1 :].strip()
    if text:
        pieces.append(text)
    return pieces


def _chunk(full_text: str, max_chars: int = 1500) -> list[tuple[str, str]]:
    """Split into size-bounded chunks, tag each with a rhetorical role.

    First split on blank-line paragraphs; then hard-split any paragraph longer
    than max_chars so we never embed a whole judgment as a single chunk (which
    would truncate to the first ~512 tokens and lose the rest)."""
    raw_paras = [p.strip() for p in re.split(r"\n\s*\n", full_text) if p.strip()]
    paras: list[str] = []
    for p in raw_paras:
        paras.extend(_split_long(p, max_chars) if len(p) > max_chars else [p])

    chunks: list[tuple[str, str]] = []
    buf = ""
    for p in paras:
        if len(buf) + len(p) > max_chars and buf:
            chunks.append((_label_role(buf), buf))
            buf = p
        else:
            buf = f"{buf}\n\n{p}" if buf else p
    if buf:
        chunks.append((_label_role(buf), buf))
    return chunks


def main() -> None:
    ap = argparse.ArgumentParser(description="Chunk + embed corpus, build FAISS index.")
    ap.add_argument("--raw", type=Path, default=RAW_DIR)
    ap.add_argument("--out", type=Path, default=INDEX_DIR)
    args = ap.parse_args()

    import faiss
    import pandas as pd

    raw_files = sorted(args.raw.glob("*.json"))
    if not raw_files:
        raise SystemExit(
            f"No raw judgments in {args.raw}. Run download_corpus.py first (Phase 0)."
        )

    tok, model, torch, device = _load_embedder()
    rows: list[dict] = []
    texts: list[str] = []

    for fp in raw_files:
        doc = json.loads(fp.read_text(encoding="utf-8"))
        for role, chunk_text in _chunk(doc.get("full_text", "")):
            rows.append(
                {
                    "id": f"{fp.stem}:{len(rows)}",
                    "case_name": doc.get("case_name", ""),
                    "citation": doc.get("citation", ""),
                    "court": doc.get("court", ""),
                    "year": doc.get("year"),
                    "outcome": doc.get("outcome", ""),
                    "segment_role": role,
                    "chunk_text": chunk_text,
                }
            )
            texts.append(chunk_text)

    avg_len = int(sum(len(t) for t in texts) / max(len(texts), 1))
    roles = {}
    for r in rows:
        roles[r["segment_role"]] = roles.get(r["segment_role"], 0) + 1
    print(f"Chunks: {len(texts)} from {len(raw_files)} judgments | avg {avg_len} chars/chunk")
    print(f"Roles: {roles}")
    if len(texts) <= len(raw_files):
        print("WARNING: ~1 chunk per judgment — chunking likely collapsed (check PDF text).")
    print(f"Embedding {len(texts)} chunks on {device}...")
    embeddings = _embed(texts, tok, model, torch, device)

    args.out.mkdir(parents=True, exist_ok=True)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)  # inner product on normalized == cosine
    index.add(embeddings)
    faiss.write_index(index, str(args.out / "corpus.faiss"))
    pd.DataFrame(rows).to_parquet(args.out / "corpus_meta.parquet", index=False)

    print(f"Wrote {index.ntotal} vectors -> {args.out/'corpus.faiss'}")
    print(f"Wrote metadata -> {args.out/'corpus_meta.parquet'}")


if __name__ == "__main__":
    main()


## 3. Download the corpus

`LIMIT_PER_YEAR` is the main scaling knob. Per-PDF S3 download is the bottleneck.
200/year × 10 years ≈ up to 2000 judgments.


In [ ]:
START_YEAR = 2015
END_YEAR   = 2024
LIMIT_PER_YEAR = 200

!python /content/download_corpus.py \
    --start-year $START_YEAR --end-year $END_YEAR \
    --limit $LIMIT_PER_YEAR \
    --out /content/data/raw


## 4. Chunk + embed + build the FAISS index (GPU)

Watch the printed chunk count + role distribution. Expect *many* chunks per
judgment (tens of thousands total) and a mix of roles — not ~1 chunk/judgment.


In [ ]:
!python /content/chunk_embed.py \
    --raw /content/data/raw \
    --out /content/data/index

import faiss, os
print('Artifacts:', os.listdir('/content/data/index'))
print('vectors:', faiss.read_index('/content/data/index/corpus.faiss').ntotal)


## 5. Download the artifacts

Unzip into the repo at `data/index/` (so `corpus.faiss` + `corpus_meta.parquet`
sit directly there), then restart the backend.


In [ ]:
import shutil
shutil.make_archive('/content/corpus_index', 'zip', '/content/data/index')
from google.colab import files
files.download('/content/corpus_index.zip')
